In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [3]:
# Load the cleaned analytical dataset created in Notebook 1
loan = pd.read_csv("data/loan_cleaned.csv", low_memory=False)

print("Shape:", loan.shape)
loan.head()

Shape: (38577, 34)


,loan_amnt,funded_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,...,total_rec_prncp,total_rec_int,recoveries,last_pymnt_amnt,default_flag,loan_year,loan_month,loan_month_name,int_rate_band,dti_band
0,5000,5000,36.0,10.65,162.87,B,B2,10+ years,RENT,24000.0,...,5000.00,863.16,0.00,171.62,0,2011,12,December,8-11.99,20-29.99
1,2500,2500,60.0,15.27,59.83,C,C4,< 1 year,RENT,30000.0,...,456.46,435.17,117.08,119.66,1,2011,12,December,12-15.99,<10
2,2400,2400,36.0,15.96,84.33,C,C5,10+ years,RENT,12252.0,...,2400.00,605.67,0.00,649.91,0,2011,12,December,12-15.99,<10
3,10000,10000,36.0,13.49,339.31,C,C1,10+ years,RENT,49200.0,...,10000.00,2214.92,0.00,357.48,0,2011,12,December,12-15.99,20-29.99
4,5000,5000,36.0,7.90,156.46,A,A4,3 years,RENT,36000.0,...,5000.00,632.21,0.00,161.03,0,2011,12,December,<8,10-19.99


In [4]:
# Check the binary target distribution
print(loan["default_flag"].value_counts())
print("\nTarget share:")
print(loan["default_flag"].value_counts(normalize=True))

default_flag
0    32950
1     5627
Name: count, dtype: int64

Target share:
default_flag
0    0.854136
1    0.145864
Name: proportion, dtype: float64


In [5]:
# Select only features that would be reasonably available at loan origination
# Exclude leakage variables that are only known after repayment behavior unfolds
feature_cols = [
    "loan_amnt",
    "funded_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "addr_state",
    "dti",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "loan_year"
]

target_col = "default_flag"

X = loan[feature_cols].copy()
y = loan[target_col].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (38577, 22)
Target shape: (38577,)


In [6]:
# Split columns into numeric and categorical groups for preprocessing
numeric_features = [
    "loan_amnt",
    "funded_amnt",
    "term",
    "int_rate",
    "installment",
    "annual_inc",
    "dti",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "loan_year"
]

categorical_features = [
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "verification_status",
    "purpose",
    "addr_state"
]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 15
Categorical features: 7


In [7]:
# Split data into training and test sets
# Stratify ensures the class balance is preserved in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (30861, 22)
Test shape: (7716, 22)


In [8]:
# Numeric preprocessing:
# 1. Fill missing values with median
# 2. Scale features for models like logistic regression
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing:
# 1. Fill missing values with most frequent category
# 2. One-hot encode categories
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine numeric and categorical preprocessing
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [9]:
# Logistic regression is a strong baseline for binary classification
log_reg_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

log_reg_model.fit(X_train, y_train)

# Generate predictions
y_pred_log = log_reg_model.predict(X_test)
y_prob_log = log_reg_model.predict_proba(X_test)[:, 1]

In [10]:
# Evaluate baseline model performance
print("Logistic Regression Performance")
print("-" * 40)
print("Accuracy:", round(accuracy_score(y_test, y_pred_log), 4))
print("Precision:", round(precision_score(y_test, y_pred_log), 4))
print("Recall:", round(recall_score(y_test, y_pred_log), 4))
print("F1 Score:", round(f1_score(y_test, y_pred_log), 4))
print("ROC AUC:", round(roc_auc_score(y_test, y_prob_log), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_log))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_log))

Logistic Regression Performance
----------------------------------------
Accuracy: 0.6599
Precision: 0.2458
Recall: 0.6444
F1 Score: 0.3559
ROC AUC: 0.7065

Confusion Matrix:
[[4367 2224]
 [ 400  725]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.66      0.77      6591
           1       0.25      0.64      0.36      1125

    accuracy                           0.66      7716
   macro avg       0.58      0.65      0.56      7716
weighted avg       0.82      0.66      0.71      7716



In [11]:
# Random Forest can capture non-linear relationships and interactions
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_split=20,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

rf_model.fit(X_train, y_train)

# Generate predictions
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

In [12]:
# Evaluate random forest performance
print("Random Forest Performance")
print("-" * 40)
print("Accuracy:", round(accuracy_score(y_test, y_pred_rf), 4))
print("Precision:", round(precision_score(y_test, y_pred_rf), 4))
print("Recall:", round(recall_score(y_test, y_pred_rf), 4))
print("F1 Score:", round(f1_score(y_test, y_pred_rf), 4))
print("ROC AUC:", round(roc_auc_score(y_test, y_prob_rf), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

Random Forest Performance
----------------------------------------
Accuracy: 0.6737
Precision: 0.2492
Recall: 0.6151
F1 Score: 0.3547
ROC AUC: 0.7068

Confusion Matrix:
[[4506 2085]
 [ 433  692]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.68      0.78      6591
           1       0.25      0.62      0.35      1125

    accuracy                           0.67      7716
   macro avg       0.58      0.65      0.57      7716
weighted avg       0.82      0.67      0.72      7716



In [13]:
# Summarize model results in a comparison table
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_log),
        accuracy_score(y_test, y_pred_rf)
    ],
    "Precision": [
        precision_score(y_test, y_pred_log),
        precision_score(y_test, y_pred_rf)
    ],
    "Recall": [
        recall_score(y_test, y_pred_log),
        recall_score(y_test, y_pred_rf)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_log),
        f1_score(y_test, y_pred_rf)
    ],
    "ROC AUC": [
        roc_auc_score(y_test, y_prob_log),
        roc_auc_score(y_test, y_prob_rf)
    ]
})

results.round(4)

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Logistic Regression,0.6599,0.2458,0.6444,0.3559,0.7065
1,Random Forest,0.6737,0.2492,0.6151,0.3547,0.7068


In [14]:
# Identify the best model based on ROC AUC
best_model_name = results.sort_values("ROC AUC", ascending=False).iloc[0]["Model"]

print(f"Best model by ROC AUC: {best_model_name}")

Best model by ROC AUC: Random Forest
